# Enseñar a dictaminar, no a resolver

### Módulo 4 · Adaptador de diagnóstico · experimento

**Proyecto:** Tutor inteligente de matemáticas · Comparación de arquitecturas
mediante fine-tuning
**Curso:** SI4006 · Tópicos Especiales y Aplicaciones en IA · Universidad EAFIT
**Notebook base:** `S04_Lab_Fine_tuning_Qwen.ipynb`

---

## 1 · Qué problema ataca

En la corrida del 24-10 el tutor **rehízo correctamente** los cinco
procedimientos con error, y no avisó de ninguno:

| El estudiante escribió | El tutor respondió | ¿Avisó? |
|---|---|---|
| `24 × 13 = 302` | `24 × 13 = 312` | no |
| `1248 ÷ 8 = 146` | `1248 ÷ 8 = 156` | no |
| `12 × 5 = 70` | `12 × 5 = 60` | no |
| `6/8 + 1/8 = 8/8` | `6/8 + 1/8 = 7/8` | no |

Los 7 casos que no emitieron `Diagnóstico:` son **exactamente** los que tenían un
error. No es un problema de aritmética ni de lectura: el adaptador de M1 se
entrenó para emitir `Paso N: … Respuesta final: …` y es lo único que sabe
producir. Ni un reintento pidiéndole solo la línea lo saca de ahí.

El modelo nunca ha visto un ejemplo de **revisar y dictaminar**. Esto lo escribe.

> **Es un experimento.** Son 33 ejemplos de entrenamiento. Lo esperable es que
> mejore el formato de salida; que además localice el paso correcto es otra cosa,
> y por eso las dos cifras se reportan **por separado** y nunca se promedian.

## 2 · Qué no toca

| | |
|---|---|
| **Adaptador nuevo** | `adaptadores/qwen-lora-diagnostico` |
| **Intactos** | `adaptadores/qwen-lora` (M1) y `adaptadores/qwen-lora-agente` (S10) |
| **Casos de evaluación** | ninguno: aquí solo hay desarrollo. Los 12 `vis-*` los mide S13 |
| **Etiquetas esperadas** | sin tocar |

Los tres adaptadores quedan en Drive para comparar y para poder volver atrás.

---
## 0 · Preparación

> Hace falta GPU con CUDA; cualquiera sirve. El entrenamiento son minutos: una
> LoRA sobre 1.5B con 33 ejemplos.

In [ ]:
%pip install -q "transformers>=4.45" "peft>=0.12" "accelerate>=0.33" datasets pandas
%pip uninstall -y -q torchao
print("Listo. Si Colab pide reiniciar la sesión, reinícienla y sigan desde aquí.")

In [ ]:
import gc, os, random
from contextlib import nullcontext

import numpy as np
import pandas as pd
import torch
import transformers
from transformers import AutoModelForCausalLM

SEMILLA = 42
random.seed(SEMILLA); np.random.seed(SEMILLA); torch.manual_seed(SEMILLA)
transformers.set_seed(SEMILLA)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"transformers {transformers.__version__} | torch {torch.__version__} | {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU detectada: {torch.cuda.get_device_properties(0).name}")
else:
    print("Sin GPU: no se puede entrenar. Activen un entorno con CUDA.")

### Persistencia entre notebooks

Los notebooks 3 y 5 **leen carpetas que producen los notebooks 1, 2 y 4**:

```
1 · Qwen    -> adaptadores/qwen-lora/      ─┐
2 · BERT    -> modelos/bert-clasificador/  ─┤-> el notebook 3 las lee
4 · FLAN-T5 -> adaptadores/flan-t5-lora/    │
1,2,3,4     -> resultados/*.json           ─┴-> el notebook 5 los lee
```

En Colab, `/content` se borra al desconectar el runtime, así que ese trabajo se
perdería entre sesiones. Montando Google Drive y trabajando desde una carpeta
suya, los artefactos sobreviven y cada notebook se puede ejecutar el día que se
pueda.

Con `USAR_DRIVE = False` todo queda en `/content`, lo cual es válido si
ejecutan los notebooks 1, 2 y 3 seguidos sin desconectar.

Fuera de Colab la celda no hace nada: el directorio de trabajo se queda como
está.

> **Los checkpoints intermedios nunca van a Drive.** El `Trainer` guarda en
> `output_dir` el modelo *más el estado del optimizador* en cada época. Para el
> notebook 2, que hace fine-tuning completo de BETO, eso son varios GB que
> además se escribirían por red. Como son desechables —lo que importa es el
> modelo final—, se mandan siempre al disco local del runtime mediante
> `DIR_CHECKPOINTS`.

In [ ]:
import os
from pathlib import Path

USAR_DRIVE    = True
CARPETA_DRIVE = "/content/drive/MyDrive/ProyectoIA"

try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

if EN_COLAB and USAR_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(CARPETA_DRIVE).mkdir(parents=True, exist_ok=True)
    os.chdir(CARPETA_DRIVE)

# Checkpoints del Trainer: grandes y desechables -> siempre en disco local.
DIR_CHECKPOINTS = "/content/salidas" if EN_COLAB else "salidas"

print(f"En Colab           : {EN_COLAB}")
print(f"Directorio de trabajo: {Path.cwd()}")
print(f"Checkpoints en     : {DIR_CHECKPOINTS}")

In [ ]:
import inspect
from transformers import TrainingArguments


def construir_args(clase=TrainingArguments, **kwargs):
    """Crea TrainingArguments filtrando los parámetros que la versión instalada
    de `transformers` no reconoce.

    Motivo: entre versiones recientes `evaluation_strategy` pasó a llamarse
    `eval_strategy`. Sin esta capa, el notebook funciona hoy y falla el
    semestre que viene. Se prefiere `eval_strategy` y se traduce si hace falta.
    """
    admitidos = set(inspect.signature(clase.__init__).parameters)
    if "eval_strategy" in kwargs and "eval_strategy" not in admitidos:
        kwargs["evaluation_strategy"] = kwargs.pop("eval_strategy")
    descartados = [k for k in kwargs if k not in admitidos]
    for k in descartados:
        kwargs.pop(k)
    if descartados:
        print(f"Parámetros no soportados por esta versión, se omiten: {descartados}")
    return clase(**kwargs)

---
## 3 · El módulo de visión

El mismo archivo que usa S13. Importa porque de aquí salen `prompt_revision` —el
prompt con el que se entrena, que tiene que ser **literalmente** el de
inferencia— y `extraer_diagnostico`, el parser con el que se mide.

In [ ]:
"""La lógica de M4 que no necesita GPU ni fotos, para poder probarla.

El notebook de M4 pega este archivo tal cual, igual que hace con `registro.py` o
`nucleo.py`. Fuera queda solo lo que no se puede probar sin el modelo cargado: la
pasada del VLM y el redimensionado de la imagen.

Dentro queda lo que puede equivocarse **en silencio**. La corrida del 22-10-2026
(archivada en `m4/corridas/corrida-01_2026-10-22/`) dio 0/11 y las cuatro causas
fueron de medición, no del modelo:

  1. **El criterio contaba menciones de pasos dentro de la explicación.** Todos
     los casos fallaron con "cayó en la trampa: 'paso 1'" porque la respuesta
     empieza repasando el procedimiento. `vis-05` llegó a decir "Paso 2:
     Incorrecto" —la respuesta esperada— y aun así se contó como fallo. Ahora el
     tutor tiene que emitir una línea `Diagnóstico:` y es **esa línea** la que se
     califica; la explicación se evalúa aparte, como señal, nunca como acierto.

  2. **La numeración que veía el tutor no coincidía con la del eval set.** El eval
     set cuenta la línea "Respuesta final" como un paso más; el VLM la devuelve en
     el campo `respuesta_final`. Para `vis-03`, cuyo error está en el paso 3, el
     tutor solo veía dos pasos: la respuesta correcta era inalcanzable. Ahora
     `pasos_numerados` reconstruye la numeración del eval set.

  3. **El inventario emparejaba por nombre exacto.** `vis-10.jpeg` no satisfacía
     `vis-10.jpg` y el caso quedó sin evaluar. Ahora se empareja por nombre base.

  4. **`lectura_fiel` exigía el mismo número de pasos.** `vis-01` y `vis-03` se
     atribuyeron a fallo de lectura cuando la transcripción era fiel y lo que
     cambiaba era dónde había puesto el VLM la respuesta final.

**Lo que no cambia:** ningún valor esperado del eval set. Los pasos, el
`paso_error` y las claves siguen siendo los que se escribieron antes de ver una
sola respuesta del modelo.

Solo biblioteca estándar: las pruebas corren sin instalar nada.
"""

from __future__ import annotations

import json
import re
import unicodedata
from pathlib import Path

# `vis-12` reutiliza a propósito la hoja de `vis-07` con otra pregunta: se pide un
# dato que no está escrito en ella. El eval set guarda el nombre lógico y el alias
# se resuelve aquí, así que hay 12 casos y 11 fotos, y el eval set no se toca.
ALIAS_IMAGENES = {"vis-12.jpg": "vis-07.jpg"}

# Extensiones que cuentan como "la foto de este caso". El eval set escribe `.jpg`,
# pero una cámara o una conversión producen `.jpeg` o `.png` sin que eso cambie
# nada del caso: lo que identifica a la foto es el nombre base.
EXTENSIONES = (".jpg", ".jpeg", ".png")


def _norm(texto: str) -> str:
    """Minúsculas y sin tildes, igual que el criterio de M2 y M3."""
    t = unicodedata.normalize("NFKD", (texto or "").lower())
    return "".join(c for c in t if not unicodedata.combining(c))


# --------------------------------------------------------------------------
# Inventario de imágenes
# --------------------------------------------------------------------------

def archivo_de(caso: dict) -> str:
    """El nombre lógico de la foto de este caso, resuelto el alias."""
    return ALIAS_IMAGENES.get(caso["imagen"], caso["imagen"])


def base_de(nombre: str) -> str:
    """`vis-10.jpeg` -> `vis-10`. El nombre base es lo que identifica la foto."""
    for ext in EXTENSIONES:
        if nombre.lower().endswith(ext):
            return nombre[: -len(ext)]
    return nombre


class ImagenAmbigua(Exception):
    """Dos archivos con el mismo nombre base, por ejemplo `vis-10.jpg` y
    `vis-10.jpeg`.

    No se elige uno por orden alfabético: si las dos fotos son distintas, la
    corrida mediría una hoja y el informe hablaría de la otra, y nada avisaría.
    Hay que borrar la que no corresponda.
    """


def inventario(casos, presentes) -> tuple[dict, dict, set]:
    """-> (pedidas, faltan, sobran).

    `pedidas` va del nombre lógico al par (archivo real en disco, casos que lo
    usan), así que una foto compartida aparece una sola vez: es lo que mantiene
    separados el recuento de fotos y el de casos.

    Empareja por nombre base, de modo que `vis-10.jpeg` satisface `vis-10.jpg`.
    Lanza `ImagenAmbigua` si dos archivos comparten el nombre base.
    """
    por_base: dict = {}
    for archivo in sorted(presentes):
        if not archivo.lower().endswith(EXTENSIONES):
            continue
        por_base.setdefault(base_de(archivo), []).append(archivo)

    ambiguas = {b: v for b, v in por_base.items() if len(v) > 1}
    if ambiguas:
        detalle = "; ".join(f"{b}: {', '.join(v)}" for b, v in sorted(ambiguas.items()))
        raise ImagenAmbigua(
            f"hay más de una imagen con el mismo nombre base ({detalle}). "
            "Dejen una sola por caso y vuelvan a ejecutar: elegir una por orden "
            "mediría una hoja y el informe hablaría de la otra.")

    pedidas: dict = {}
    for c in casos:
        logico = archivo_de(c)
        real = (por_base.get(base_de(logico)) or [None])[0]
        entrada = pedidas.setdefault(logico, {"archivo": real, "casos": []})
        entrada["casos"].append(c["id"])

    faltan = {k: v for k, v in pedidas.items() if v["archivo"] is None}
    usados = {v["archivo"] for v in pedidas.values() if v["archivo"]}
    sobran = {a for lista in por_base.values() for a in lista} - usados
    return pedidas, faltan, sobran


# --------------------------------------------------------------------------
# El JSON que devuelve el VLM
# --------------------------------------------------------------------------

def json_de(texto: str):
    """El primer objeto JSON del texto, o None.

    Un modelo de 2B devuelve el JSON envuelto en prosa o en una valla de código
    (pasó con `vis-06`): `json.loads` del texto completo fallaría y se perdería
    una lectura buena.
    """
    i, j = (texto or "").find("{"), (texto or "").rfind("}")
    if i == -1 or j <= i:
        return None
    try:
        return json.loads(texto[i:j + 1])
    except json.JSONDecodeError:
        return None


_PREFIJO_PASO = re.compile(r"^\s*paso\s*\d+\s*[:.\-]\s*", re.IGNORECASE)
_MENCIONA_FINAL = re.compile(r"respuesta\s*final", re.IGNORECASE)


def pasos_numerados(lectura) -> list:
    """Los pasos de la hoja con la MISMA numeración que usa el eval set.

    El eval set cuenta la línea "Respuesta final: ..." como un paso más —el error
    de `vis-03` está justamente ahí—. El VLM a veces la devuelve dentro de
    `pasos` y a veces en el campo `respuesta_final`. Sin unificarlo, el paso que
    hay que señalar no existe en lo que el tutor ve, y el caso es imposible de
    acertar por construcción.

    También quita el "Paso N:" que el VLM escribe a veces dentro del texto, para
    no numerar dos veces.
    """
    js = (lectura or {}).get("json") or {}
    pasos = [_PREFIJO_PASO.sub("", str(p)).strip() for p in (js.get("pasos") or [])]
    final = js.get("respuesta_final")
    # Solo se añade la respuesta final si HAY pasos. Con `pasos: []` —la lista de
    # mercado de `vis-11`— añadirla fabricaba un "Paso 1: Respuesta final: Arroz"
    # y al tutor se le presentaba un procedimiento de un paso donde la hoja no
    # tenía ninguno, lo que hace casi imposible concluir "fuera de dominio".
    if pasos and final is not None and not any(_MENCIONA_FINAL.search(p) for p in pasos):
        pasos.append(f"Respuesta final: {final}")
    return pasos


# --------------------------------------------------------------------------
# ¿Es fiable esta lectura?
# --------------------------------------------------------------------------

_PAT_NUMERO = re.compile(r"-?\d+(?:[.,]\d+)?(?:/\d+)?")


def numeros_de(texto: str) -> list:
    """Los números de un texto, normalizados. La huella de una transcripción."""
    return [n.replace(",", ".") for n in _PAT_NUMERO.findall(texto or "")]


def huella_lectura(lectura) -> tuple:
    """Lo que tiene que coincidir entre dos lecturas de la MISMA foto: cuántos
    pasos y qué números. La redacción puede variar; los números, no."""
    js = (lectura or {}).get("json") or {}
    if js.get("ilegible"):
        return ("ilegible",)
    pasos = js.get("pasos") or []
    texto = " ".join(map(str, pasos)) + " " + str(js.get("respuesta_final", ""))
    return (len(pasos), tuple(numeros_de(texto)))


def lecturas_consistentes(a, b) -> bool:
    """¿Dos lecturas de la misma foto dicen lo mismo?

    **Necesaria pero NO suficiente, y hay prueba.** En la corrida del 24-10 esta
    señal falló exactamente donde tenía que servir: ante la foto desenfocada de
    `vis-10` el VLM inventó `24 x 13 = 306` (la hoja dice 302) y lo inventó **las
    dos veces**. La invención no era ruido de muestreo: estaba anclada en el
    prior del modelo —la hoja es una versión borrosa de un ejercicio de 24 × 13—
    y por eso es estable.

    Se conserva porque es barata y sí caza las lecturas inestables, pero ya no se
    usa como única defensa.
    """
    return huella_lectura(a) == huella_lectura(b)


def atipicos_por_mad(valores: dict, k: float = 3.0) -> set:
    """Las claves cuyo valor queda muy por debajo del resto del conjunto.

    Mediana y desviación absoluta mediana, el criterio robusto de siempre:
    `valor < mediana - k · MAD`. Es **relativo al propio conjunto**, no un umbral
    absoluto elegido a mano, y eso es deliberado: cualquier constante calibrada
    con estas 11 fotos sería un umbral ajustado al conjunto de prueba.

    `k = 3` es la convención estadística habitual, no un valor afinado para que
    `vis-10` caiga del lado correcto. **Si separa o no esa foto es algo que hay
    que medir, y no debe tocarse para que salga.**
    """
    if len(valores) < 4:          # con tres o menos no hay "resto del conjunto"
        return set()
    import statistics
    xs = sorted(valores.values())
    mediana = statistics.median(xs)
    mad = statistics.median([abs(x - mediana) for x in xs]) * 1.4826
    if mad == 0:
        return set()
    return {clave for clave, v in valores.items() if v < mediana - k * mad}


# Dos ideas tentadoras que NO se usan, y por qué:
#
#   · **Comprobar la aritmética de la transcripción.** Suena bien y es inservible
#     aquí: las hojas llevan cuentas mal A PROPÓSITO. En `vis-10` el invento es
#     `24 × 13 = 306` y lo que la hoja dice es `24 × 13 = 302`; las dos son
#     aritméticamente falsas, así que la comprobación no distingue la lectura de
#     la invención. Peor: marcaría como dudosas justo las hojas con error, que son
#     las que el módulo existe para medir.
#
#   · **Un umbral de nitidez sobre la imagen** (varianza del laplaciano y
#     similares). Habría que calibrarlo, y calibrarlo con estas 11 fotos es
#     ajustarlo al conjunto de prueba. Si algún día hay un conjunto de calibración
#     aparte, vuelve a ser una opción razonable.


def lectura_no_fiable(lectura) -> tuple:
    """-> (no_fiable, motivo). Si es no fiable, el tutor NO debe tratarla como un
    procedimiento legible: se le dice que la hoja no se pudo leer.

    Cuatro señales de naturaleza distinta, porque la del 24-10 demostró que una
    sola no basta. Basta que **una** se dispare: ante la duda no se le entrega al
    tutor un procedimiento que puede ser inventado.

      1. la transcripción no salió estructurada;
      2. el propio VLM declaró la hoja ilegible al transcribir;
      3. preguntado **aparte** si la hoja se lee —sin transcribir, otra pregunta y
         otro encuadre, para no arrastrar el prior de la transcripción—, dijo que
         no (`legible_declarada = False`);
      4. la confianza del modelo en los **números** que transcribió es atípica
         frente a las demás fotos del conjunto (`confianza_atipica`). Un número
         leído de una cifra nítida sale con más confianza que uno rellenado.

    La señal de consistencia entre dos generaciones queda como quinta, degradada:
    sirve, pero no se le confía la decisión.

    Nunca acredita un caso: una lectura no fiable no vuelve correcta ninguna
    respuesta, solo evita juzgar al tutor sobre un invento del VLM.
    """
    if not lectura or not lectura.get("ok"):
        return True, "la transcripción no salió estructurada"
    js = lectura.get("json") or {}
    if js.get("ilegible"):
        return True, js.get("motivo") or "el modelo la declaró ilegible"
    if lectura.get("legible_declarada") is False:
        return True, ("preguntado aparte, el modelo dijo que la hoja no se lee: "
                      + (lectura.get("motivo_legibilidad") or "sin motivo"))
    if lectura.get("confianza_atipica") is True:
        return True, ("la confianza del modelo en los números transcritos es "
                      "atípicamente baja frente a las demás fotos")
    if lectura.get("consistente") is False:
        return True, ("dos lecturas de la misma foto no coinciden: la transcripción "
                      "no es de fiar")
    return False, ""


# --------------------------------------------------------------------------
# El diagnóstico: una conclusión explícita, separada de la explicación
# --------------------------------------------------------------------------

# Las cinco categorías, ajustadas al esquema real del eval set:
#   paso N            -> bloques `localizacion` y `compensado` (tienen `paso_error`)
#   sin error         -> bloque `sin_error`
#   ilegible          -> adversarial: la hoja no se puede leer (vis-10)
#   fuera de dominio  -> adversarial: la hoja no es matemáticas (vis-11)
#   dato no disponible-> adversarial: se pide algo que no está escrito (vis-12)
CATEGORIAS = ("paso", "sin error", "ilegible", "fuera de dominio", "dato no disponible")

# Las tres formas de abstenerse. Cualquiera de ellas es "no inventó nada", que es
# la conducta que el bloque adversarial busca; acertar la categoría exacta es más
# exigente y se reporta por separado.
ABSTENCIONES = ("ilegible", "fuera de dominio", "dato no disponible")

# Qué abstención corresponde a cada caso adversarial. Va por id porque la
# distinción está en el enunciado del caso, no en un campo del eval set; la
# prueba `test_todo_adversarial_tiene_categoria` falla si se añade un caso
# adversarial y se olvida ponerlo aquí.
DIAGNOSTICO_ADVERSARIAL = {
    "vis-10": "ilegible",             # foto movida y desenfocada a propósito
    "vis-11": "fuera de dominio",     # una lista de mercado, no un procedimiento
    "vis-12": "dato no disponible",   # se pide una nota que no está en la hoja
}

# Sinónimos que se aceptan como la misma categoría. Un modelo pequeño no va a
# escribir la etiqueta exacta siempre, y penalizarlo por el sinónimo mediría
# obediencia al formato en vez de criterio.
_SINONIMOS = {
    "sin error": ("sin error", "sin errores", "no hay error", "no hay errores",
                  "correcto", "todo correcto", "ninguno"),
    "ilegible": ("ilegible", "no se puede leer", "no legible", "borrosa", "no se lee"),
    "fuera de dominio": ("fuera de dominio", "no es matematicas", "no es un procedimiento",
                         "no es un ejercicio", "fuera de alcance"),
    "dato no disponible": ("dato no disponible", "dato no escrito", "no aparece",
                           "no esta en la hoja", "no disponible", "no tengo esa informacion"),
}

_PAT_DIAGNOSTICO = re.compile(r"diagn[oó]stico\s*:\s*(.+?)\s*(?:\n|$)", re.IGNORECASE)
# Acepta "Paso 2", "paso n.º 2" (que `_norm` deja en "paso n.o 2") y
# "paso número 2". Un modelo pequeño escribe el ordinal de varias formas y
# penalizarlo por eso mediría obediencia al formato, no criterio.
_PAT_PASO = re.compile(
    r"paso\s*(?:(?:n[.\u00ba\u00b0o]*|numero|num\.?|#)\s*)?(\d+)", re.IGNORECASE)


def diagnostico_esperado(caso: dict) -> str:
    """La conclusión que este caso espera, derivada del eval set tal como está.

    No se toca ningún valor esperado: `paso_error` y `bloque` ya estaban escritos
    antes de ver una respuesta del modelo.
    """
    if caso.get("paso_error") is not None:
        return f"paso {caso['paso_error']}"
    if caso.get("bloque") == "sin_error":
        return "sin error"
    if caso.get("bloque") == "adversarial":
        try:
            return DIAGNOSTICO_ADVERSARIAL[caso["id"]]
        except KeyError:
            raise KeyError(
                f"el caso adversarial {caso['id']!r} no tiene categoría en "
                "DIAGNOSTICO_ADVERSARIAL: añádanla antes de evaluarlo") from None
    raise ValueError(f"no sé qué espera el caso {caso.get('id')!r}")


def extraer_diagnostico(respuesta: str) -> str | None:
    """La conclusión de la línea `Diagnóstico:`, normalizada. None si no la hay.

    **Solo mira esa línea.** Si el tutor no la emite, el caso no se cuenta como
    acierto aunque la explicación acabe diciendo lo correcto: una conclusión que
    hay que adivinar no sirve a un estudiante, y un criterio que la adivina mide
    la suerte del parser.
    """
    m = _PAT_DIAGNOSTICO.search(respuesta or "")
    if not m:
        return None
    crudo = _norm(m.group(1)).strip(" .;")
    paso = _PAT_PASO.search(crudo)
    if paso:
        return f"paso {int(paso.group(1))}"
    for canonica, variantes in _SINONIMOS.items():
        if any(v in crudo for v in variantes):
            return canonica
    return crudo or None


def diagnostico_valido(conclusion) -> bool:
    """¿La conclusión es una de las cinco categorías, y no texto libre?

    `vis-04` emitió "Diagnóstico: equivoco el producto 1248…": una línea que
    señalaba el error sin nombrar el paso. Eso no es una conclusión válida, pero
    tampoco es una invención, y contarlo como alucinación exageraba el fallo.
    """
    if not conclusion:
        return False
    return conclusion.startswith("paso ") or conclusion in ("sin error",) + ABSTENCIONES


def pasos_mencionados(texto: str) -> set:
    """Los números de paso que menciona un texto. Señal, nunca criterio."""
    return {int(n) for n in _PAT_PASO.findall(texto or "")}


def _sin_la_linea_de_diagnostico(respuesta: str) -> str:
    return _PAT_DIAGNOSTICO.sub("", respuesta or "")


def calificar_m4(caso: dict, respuesta: str) -> dict:
    """Califica un caso de M4 leyendo la conclusión, no la explicación.

    Devuelve señales independientes, igual que `evaluar_caso` en M2 y M3:

      acierto       la conclusión coincide con la que el caso espera
      abstuvo       concluyó una de las tres formas de "no puedo / no está"
      abstuvo_bien  se abstuvo **y** era un caso en que había que abstenerse
      alucino       concluyó una categoría válida, concreta y equivocada
      sin_conclusion no emitió la línea `Diagnóstico:`
      conclusion_invalida  la emitió, pero en texto libre que no es ninguna de
                           las cinco categorías

    Y, por separado, sobre la explicación —que se conserva y se evalúa, pero no
    decide el acierto—:

      explicacion_pasos   qué pasos menciona la explicación
      explicacion_justifica  si la explicación habla del paso que concluyó (o no
                             menciona ninguno). No es un juicio de calidad: dice
                             si la conclusión viene argumentada en el texto.
    """
    esperado = diagnostico_esperado(caso)
    obtenido = extraer_diagnostico(respuesta)
    explicacion = _sin_la_linea_de_diagnostico(respuesta)
    mencionados = pasos_mencionados(explicacion)

    valida = diagnostico_valido(obtenido)
    acierto = valida and obtenido == esperado
    abstuvo = obtenido in ABSTENCIONES
    debia_abstenerse = esperado in ABSTENCIONES

    if obtenido is None:
        motivo = "no emitió la línea 'Diagnóstico:'"
    elif not valida:
        motivo = f"conclusión no válida: {obtenido[:48]!r}"
    elif acierto:
        motivo = f"concluyó {obtenido!r}"
    else:
        motivo = f"esperaba {esperado!r} y concluyó {obtenido!r}"

    paso_concluido = (int(obtenido.split()[1]) if obtenido and obtenido.startswith("paso ")
                      else None)
    return {
        "acierto": acierto,
        "conclusion": obtenido,
        "esperado": esperado,
        "abstuvo": abstuvo,
        "abstuvo_bien": abstuvo and debia_abstenerse,
        # Alucinar es concluir algo CONCRETO y equivocado. No emitir conclusión, o
        # emitirla en texto libre, es un fallo de formato y se cuenta aparte: son
        # problemas distintos y se arreglan de forma distinta.
        "alucino": valida and (not acierto) and not abstuvo,
        "sin_conclusion": obtenido is None,
        "conclusion_invalida": (obtenido is not None) and not valida,
        "motivo": motivo,
        "explicacion_pasos": sorted(mencionados),
        "explicacion_justifica": (paso_concluido in mencionados if paso_concluido
                                  else not mencionados),
    }


# --------------------------------------------------------------------------
# Adaptadores: comprobar la ruta de verdad
# --------------------------------------------------------------------------

# Qué tiene que haber dentro de un adaptador LoRA guardado con `save_pretrained`.
ARCHIVOS_ADAPTADOR = ("adapter_config.json", "adapter_model.safetensors")


def revisar_adaptador(ruta, requeridos=ARCHIVOS_ADAPTADOR) -> tuple:
    """-> (ok, motivo). Mira el disco y dice **qué** falta, si falta algo.

    Por qué existe. La corrida del 25-10 imprimió "No está
    adaptadores/qwen-lora-diagnostico" con los dos archivos presentes en Drive:
    el mensaje afirmaba un hecho del sistema de archivos que el código nunca
    había comprobado —la condición solo miraba si el adaptador estaba *cargado*,
    y nadie lo cargaba—. Un aviso que no verifica lo que afirma manda a buscar el
    fallo donde no está, y costó una corrida entera.
    """
    p = Path(ruta)
    if not p.exists():
        return False, f"no existe el directorio {p}"
    if not p.is_dir():
        return False, f"{p} existe pero no es un directorio"
    faltan = [n for n in requeridos if not (p / n).exists()]
    # `save_pretrained` escribe .safetensors; los adaptadores antiguos traen .bin.
    if faltan == ["adapter_model.safetensors"] and (p / "adapter_model.bin").exists():
        return True, f"{p}: pesos en adapter_model.bin (formato antiguo)"
    if faltan:
        hay = sorted(f.name for f in p.iterdir()) or ["(vacío)"]
        return False, f"en {p} faltan {faltan}; lo que hay: {hay}"
    return True, f"{p}: completo"


def elegir_adaptador(disponibles, preferido: str, alternativo: str) -> str:
    """El preferido si está cargado; si no, el alternativo. Sin inventar motivos."""
    return preferido if preferido in disponibles else alternativo


# --------------------------------------------------------------------------
# La puerta del ensayo de desarrollo
# --------------------------------------------------------------------------

class PuertaNoSuperada(RuntimeError):
    """El ensayo de formato sobre los casos `dev-*` no se superó, o no se ejecutó.

    Por qué es una excepción y no un aviso: en la corrida del 24-10 el ensayo dio
    **0/3** y la celda imprimió "PAREN AQUÍ"… y la ejecución siguió hasta evaluar
    los 12 casos. Un `print` no detiene nada. El resultado de esos 12 estaba
    decidido antes de empezar: 7 de 12 sin conclusión.
    """


def revisar_puerta_dev(conclusiones) -> dict:
    """`conclusiones`: iterable de (id, conclusión emitida). -> dict de la puerta.

    La puerta se supera si **todos** los casos de desarrollo dan una conclusión
    válida. No se exige que acierten: lo que se comprueba es el formato, porque
    es la condición sin la cual el scorecard de los 12 no mide el criterio del
    tutor sino su obediencia al formato de salida.
    """
    filas = [(cid, c, diagnostico_valido(c)) for cid, c in conclusiones]
    validos = sum(1 for _, _, ok in filas if ok)
    return {"filas": filas, "validos": validos, "total": len(filas),
            "ok": len(filas) > 0 and validos == len(filas)}


def exigir_puerta_dev(puerta, permitir_sin_puerta: bool = False) -> None:
    """Lanza `PuertaNoSuperada` si la puerta no se superó o no se ejecutó.

    Va al principio de las celdas de evaluación, así que la parada no depende de
    cómo el entorno trate una excepción en otra celda: si alguien ejecuta la
    celda de los 12 casos directamente, se detiene igual.

    `permitir_sin_puerta=True` es una salida deliberada para recoger datos de
    diagnóstico. No se silencia: quien la usa queda registrado en los artefactos
    (`puerta_dev: "no superada"`), de modo que cualquier cifra que salga de esa
    corrida lleva la marca encima.
    """
    if permitir_sin_puerta:
        return
    if puerta is None:
        raise PuertaNoSuperada(
            "el ensayo de formato sobre los casos de desarrollo no se ha ejecutado. "
            "Ejecuten la celda 7.1 antes de evaluar los 12 casos: sin ella no se "
            "sabe si el tutor emite la línea `Diagnóstico:`, y un scorecard sin "
            "eso mide el formato de salida y no el criterio del tutor.")
    if not puerta["ok"]:
        detalle = ", ".join(f"{cid}->{c!r}" for cid, c, ok in puerta["filas"] if not ok)
        raise PuertaNoSuperada(
            f"el ensayo de formato dio {puerta['validos']}/{puerta['total']} en los "
            f"casos de desarrollo ({detalle}). La evaluación de los 12 casos se "
            "detiene aquí a propósito: con el formato roto su total no mediría el "
            "criterio del tutor.\n"
            "Lo que toca es el prompt o el entrenamiento, y la decisión se toma con "
            "los casos de desarrollo, no con los 12: afinar contra los 12 los "
            "convierte en conjunto de entrenamiento.\n"
            "Si quieren correrlos igualmente para recoger diagnóstico, definan "
            "PERMITIR_SIN_PUERTA = True en una celda nueva; el scorecard quedará "
            "marcado como obtenido sin superar la puerta.")


# --------------------------------------------------------------------------
# Instrucciones
# --------------------------------------------------------------------------

INSTRUCCION_VER = (
    "Transcribe EXACTAMENTE lo que está escrito a mano en esta hoja. No corrijas "
    "ningún error: si una cuenta está mal, cópiala mal, tal como aparece.\n"
    "Devuelve SOLO un JSON con esta forma:\n"
    '{"enunciado": "...", "pasos": ["...", "..."], "respuesta_final": "..."}\n'
    "Un elemento de `pasos` por cada línea escrita.\n"
    "Si la hoja no se puede leer, o no contiene un procedimiento matemático, "
    'devuelve {"ilegible": true, "motivo": "..."}.'
)

# El formato de salida, y la evidencia que lo decidió.
#
# En las once respuestas completas de la corrida del 22-10, OCHO terminan
# copiando la última etiqueta que el prompt les mostró:
#
#     vis-01 -> "Respuesta final escrita: 24 x 13 lápices"
#     vis-03 -> "Respuesta final escrita: 36000"
#     vis-05 -> "Respuesta final escrita: 60 metros cuadrados"
#
# El modelo no desobedecía: estaba **continuando el formato de la transcripción**,
# que es lo que la LoRA de M1 le enseñó a hacer con un texto que empieza por
# "Paso 1:" y acaba en "Respuesta final:". De ahí las tres decisiones:
#
#   1. La exigencia del formato va **al final**, pegada a la generación, para que
#      lo último que el modelo lea sea la conclusión que se le pide y no la
#      respuesta final del estudiante.
#   2. Un ejemplo resuelto, porque un modelo de 1.5B copia formatos mejor de lo
#      que sigue instrucciones abstractas.
#   3. Se le dice explícitamente que NO escriba "Respuesta final", que es la
#      etiqueta que lo descarrila.
#
# Nada de esto se ajustó a los 12 casos: el ejemplo usa un ejercicio inventado
# (pan y panes), no una de las hojas, y las cinco categorías salen del esquema del
# eval set, no de lo que el modelo respondió.

INSTRUCCION_REVISION = (
    "Eres un tutor de matemáticas. Un estudiante te muestra el procedimiento que "
    "escribió a mano, ya transcrito. Tu trabajo NO es resolver el ejercicio ni "
    "reescribirlo: es revisarlo y dar un veredicto.\n"
    "Reglas:\n"
    "- Señala UN solo paso, el primero que esté mal. Si los siguientes arrastran "
    "ese error, no los cuentes aparte.\n"
    "- Si el procedimiento está correcto, dilo. No inventes un error para tener "
    "algo que decir.\n"
    "- No cambies los números del estudiante por los correctos: importa dónde se "
    "equivocó, no cuál era el resultado bueno.\n"
    "- La respuesta final escrita cuenta como un paso más, el último.\n"
    "- NO escribas 'Respuesta final': eso es lo que escribió el estudiante, no tu "
    "veredicto."
)

# NO hay ejemplo resuelto, y la corrida del 24-10 explica por qué.
#
# El ejemplo que había —una panadería, 15 panes por bandeja— apareció COPIADO en
# la respuesta a `vis-11`, la lista de mercado:
#
#     Paso 1: Multiplicamos las bandejas por los panes de cada una: 15 × 4.
#     Paso 2: 15 × 4 = 60.
#     Diagnóstico: Paso 2
#
# Copió el ejercicio entero y el veredicto del ejemplo, y "corrigió" su 54 a 60.
# Un modelo que ante una entrada degenerada se agarra a lo último parecido que vio
# convierte cualquier ejemplo resuelto en una respuesta posible.
#
# Así que del ejemplo queda solo la FORMA de la última línea, sin ningún
# procedimiento que copiar. Enseñar el formato no exige enseñar un ejercicio.
EJEMPLO_REVISION = (
    "La última línea de tu respuesta tiene que verse exactamente así, con una de "
    "las cinco etiquetas:\n"
    "  Diagnóstico: Paso 2"
)

# Esto va LO ÚLTIMO, inmediatamente antes de generar.
RECORDATORIO_FORMATO = (
    "Termina SIEMPRE con una última línea con este formato exacto, elige una:\n"
    "  Diagnóstico: Paso N              (un paso está mal; N es su número)\n"
    "  Diagnóstico: sin error           (todos los pasos están bien)\n"
    "  Diagnóstico: ilegible            (la hoja no se pudo leer)\n"
    "  Diagnóstico: fuera de dominio    (la hoja no es un ejercicio de matemáticas)\n"
    "  Diagnóstico: dato no disponible  (piden algo que no está escrito en la hoja)\n"
    "Esa línea es obligatoria, también cuando el procedimiento esté correcto."
)

# El reintento: si no hay conclusión válida, se pide SOLO la línea. No se le
# permite recalcular ni cambiar de opinión sobre el fondo —se le devuelve su
# propia explicación y se le pide que la resuma en una de las cinco etiquetas—,
# y queda registrado cuáles necesitaron reintento, porque un sistema que necesita
# dos pasadas para contestar no es lo mismo que uno que contesta de una.
PETICION_CONCLUSION = (
    "Esta fue tu revisión:\n"
    "---\n{explicacion}\n---\n"
    "No terminaste con la línea obligatoria. Escribe AHORA solo esa línea, sin "
    "ninguna explicación, coherente con lo que acabas de decir:\n"
    "  Diagnóstico: Paso N  |  Diagnóstico: sin error  |  Diagnóstico: ilegible  |  "
    "Diagnóstico: fuera de dominio  |  Diagnóstico: dato no disponible"
)


def prompt_revision(pregunta: str, lectura) -> str:
    """El prompt completo, con la exigencia del formato al final."""
    return (f"{INSTRUCCION_REVISION}\n\n{EJEMPLO_REVISION}\n\n"
            f"Ahora la hoja de verdad.\n{pasos_como_texto(lectura)}\n\n"
            f"Pregunta del estudiante: {pregunta}\n\n{RECORDATORIO_FORMATO}")


def asegurar_diagnostico(respuesta: str, reintentar) -> tuple:
    """-> (respuesta, reintento_usado).

    `reintentar(prompt) -> texto`. Se llama UNA sola vez y solo si la respuesta no
    trae una conclusión válida. La línea que devuelva se añade al final de la
    explicación original, que se conserva entera.
    """
    if diagnostico_valido(extraer_diagnostico(respuesta)):
        return respuesta, False
    linea = reintentar(PETICION_CONCLUSION.format(explicacion=(respuesta or "").strip()[:1200]))
    return f"{(respuesta or '').rstrip()}\n{(linea or '').strip()}", True


def pasos_como_texto(lectura) -> str:
    """La transcripción, en el formato que ve el tutor.

    La numeración la pone esta función, con la convención del eval set (la
    respuesta final es el último paso), y no la copia del VLM: el criterio es
    "Paso N", así que si la numeración dependiera de cómo el modelo partió las
    líneas, el caso se podría perder por una coma.
    """
    no_fiable, motivo = lectura_no_fiable(lectura)
    if no_fiable:
        return (f"(la hoja NO se pudo leer: {motivo}. No hay un procedimiento "
                "legible que revisar.)")
    js = lectura.get("json") or {}
    pasos = pasos_numerados(lectura)
    if not pasos:
        return (f"Lo único que se alcanza a leer en la hoja: "
                f"{js.get('enunciado') or '(nada)'}\n"
                "(no hay ningún paso de un procedimiento matemático)")
    lineas = [f"Enunciado: {js.get('enunciado') or '(no transcrito)'}"]
    lineas += [f"Paso {i}: {p}" for i, p in enumerate(pasos, 1)]
    return "\n".join(lineas)


# --------------------------------------------------------------------------
# Lectura o diagnóstico: a qué etapa se atribuye el fallo
# --------------------------------------------------------------------------

def lectura_fiel(caso: dict, lectura) -> bool | None:
    """¿La transcripción conserva el procedimiento que la hoja tenía escrito?

    Devuelve None cuando la pregunta no aplica (los adversariales no tienen un
    procedimiento que transcribir). None no es "sí" ni "no": un caso así nunca se
    atribuye a la lectura, porque no habría nada que leer.

    Compara sobre los pasos **ya numerados** con la convención del eval set. En la
    corrida del 22-10, exigir el mismo número de elementos en `pasos` marcó
    `vis-01` y `vis-03` como fallos de lectura cuando la transcripción era fiel y
    lo único distinto era dónde había puesto el VLM la respuesta final.
    """
    # Una lectura no fiable es un fallo de lectura, aunque el caso sea adversarial:
    # es lo que hace que `vis-10` se atribuya a la etapa de ver y no al tutor.
    if lectura_no_fiable(lectura)[0]:
        return False
    if caso.get("bloque") == "adversarial" or not caso.get("pasos"):
        return None
    js = (lectura or {}).get("json") or {}
    pasos = pasos_numerados(lectura)
    if len(pasos) != len(caso["pasos"]):
        return False
    esperado = caso.get("valor_final_escrito")
    if esperado is None:
        return True
    # El valor escrito a mano tiene que seguir ahí: es el error que el tutor debe
    # encontrar. Si el VLM lo "arregló", el caso ya no mide lo que decía medir.
    escrito = " ".join(pasos).replace(" ", "")
    return esperado.replace(" ", "") in escrito


def fuente_del_fallo(acierto: bool, fiel: bool | None) -> str:
    """A qué etapa se le atribuye un fallo. El resultado principal de M4."""
    if acierto:
        return "—"
    return "lectura" if fiel is False else "diagnostico"

---
## 4 · Las trayectorias de diagnóstico

Generadas y validadas por `scripts/trayectorias_diagnostico_fuente.py`. Ningún
enunciado sale del eval set de M2, M3 ni M4, ni parafraseado: la comprobación es
la misma de S10 (Jaccard sobre palabras, tope 0.6) y el generador falla si alguna
lo pasa. Incluye los casos `dev-*` de M4, que son la puerta de S13.

El paso señalado se reparte a propósito entre 1, 2 y 3: el eval set está sesgado
hacia "Paso 2" (4 de 6), y entrenar con el mismo sesgo enseñaría el atajo.

In [ ]:
import base64, gzip, hashlib, json
from pathlib import Path

_RUTA_DIAGNOSTICO = Path("data/trayectorias_diagnostico.jsonl")
_SHA_DIAGNOSTICO = "87bd8f19b983369395cc9a9e29c62b1e211776c74539258a62e2bb58d5147589"
_BLOB_DIAGNOSTICO = (
    "H4sIAAAAAAAC/+2dTY/cNprH7/spiL7MDNxxqvRSVd3BYLDYMbALeLxZOzltBgZbUlcro5IqUqkRJ8h9znPJ2cccfAh8GGAu"
    "A2x9k/kk+39I6l1UV78kdtlPYCBdeqEo8uGfPz0kH35/Eocn5+IkXH8iZ/OTU3FyKTdxEks6uJVF9lLm8W4T7eIgo7PFNol3"
    "dG6XyzilI2Es12lWqAtw/HPcIxw6oe6O8jzLcdyhA3m0LtOdSvp5dB0XUmxisc2zIArjTRylu+xc/N8/o1R8U+5/EnS/2EQi"
    "+qaMr7Ng/9MfKNUoLdMglqF62JepiIpdGVxFAvmJxHwlgizJ8qh4jIT+o9y/RppFdUxcyVcCqS+rm4o/VPkskNr/nvypTHYx"
    "XjCQm6w4p9T2P4rlY7rI/C1+L+bzhTryPCq2JRKS4jJOZXJOJ07+TAVIPyl3dAC/sy1+fH9Crz1f4fcF/lhWx0++VS9VBChl"
    "eiXc8wMObKK0kF9HKlvfn+RZEtG1xatiF23ohiBLdygvOviE3qxMxa7cZbkII7GRuGj/GhUiC5RkJHJd1lFChXQhv86EeR5d"
    "XeoiRC0iRYHy2cQ7dfQ6ylEtwS57fIIcNZkoiyi/TRYeiy87j8C/DZVbrnLUqX2VW5W3i3j/VsA8ZJqdileSMp7qPD8WX5T1"
    "ezz7b1yO9yuyBNml9KKvozyIgzgTaYwTJrE8gWWpK6kk8AsvGsq8+5pfpc+jdSKL86/ST8SLaP+zTKT48plA4pkyxlOd4XgT"
    "5VVWd7DTjUweixexSGBpRbwu6VXwLJnnkt6SXh6XUjs4FWmmLgtKc81W5rvosXpgPCwOSv81rDfPI2TwVIRxgvd/lok4vdYJ"
    "UO1RysgfyhP1gVKQyVpnL4yCOFeJ4xbY9EWMO+jx6f4f9A4FrkjaVbNFQip75oloA/EGB2Hj4f5tiootmub4Vr1NgDaGNHRl"
    "oiLQgtA2xQVeMFOPfirpcLulGOuTuhTwv2yT0Yvo9r5/Xahy3v8DbXGj01DVTBUJg/5Nr939hiqWigo579lP5+VUbndlp76/"
    "SpE7/SApkv2bNJJkvbiqyfIOVRGphHEnvf63MtjJDRW/kMX+zSnynyL76k5YjwjiNEB+djHu2SlrEuKPpJH7t0okz4WWSHr8"
    "v19lKDmY2VX2tUoAzwhliKw9qWTuXNxb5L5K1RPn58IicOYCpz5iZM4cd8/FmNjRC3xuJL1nSXTDHfSdUvwiyjd4hHjxX0/+"
    "9PnzJ3Xp9qqJDuN5kbjMcqhNputFNdF4TfolbQX/THT++21lebqxoTF/hktQjkVZtZPfDVMq4qrhtVLaZWGmG5jqUnSKqUCz"
    "S0eSiJNoHV8kUSczlSnAVmFr2xJNKYmifOT2y5IaHUwmzFBe0Lve7UYcajnsqfJIiiEVI+4M42KbpTprv93GIeqrlpS0Kqeq"
    "B8FJ81Ck+KSo6wePzy5QFUgzj+Wp2JH87H9K0ehlirca1bqfauXp9TmyKGIYH3qbXseT6Lpzmto7p/xsajsPIARpbdahFHNn"
    "oaRA2/dYwzz54c8//Nv3bS5yHoqL5gMumv9CXCTFRRZGaynWpYSkiLPFDEd2UUIShZTmqsik6kTQZ8mkrSa4hLoCc0EPkp7T"
    "YSUflOa//vo3pKWAqPkJ+TjzVuOUhBNdSqIDFQ2lZZIw/TD9MP0w/dT085BS1qDQqIw1IDQQswkWwmlmIWah94SF5mKbkN2p"
    "lpVtUTKagYxpyVCeCwmje6O+FJRSxtcxnp0VSiVy0zDG6Wg+pCP3oejIHdCR+0t5jcJoi/eiStMK53izmUAeqVt4pXT+W/rW"
    "SsVq6Xe+slBeoUytSKTSId2g++ii9gH6qvId3+I9Wjh+z31ER7r+I0rNeJCQXu1D+qTnQ8KN7ERijGKMYoxqOZHuq3lj7NTR"
    "u4aexlRvypkEwWKCYoJ6xwT1lMSMylur80jBn4otyXZkUMvFE7Yx9TBJR7nOlZaR1QtFWCHpiHlFsvVxtHKHaOUdm+PpiW5e"
    "0bdBmRcKO6/RdzkuOtL9z0pn5mfqb9l1XlMXBsPa6q84gOn0yBwS3P+IpDRimR8QGs9djtMVTnThig6w14lxiXGJcWkElx5O"
    "x6yDb7WGtbCpq2QTxITTDEwMTEfrcirKjawcTpt2uzjY6+Qf35gcriJ9ok4CcEhVEUZ5qrsmatueM8Nn0xbS2haV6iwVY32P"
    "1QGl01Cea+1oah8gXZmfrSweKHWqR0nqEHMScxJzEnPSuFvpAUVtzMPUEbQGlUZlbQqY6AJGJkamj3WUbnELXgqj63c/RidJ"
    "36n7hNEUMpDCWYjt/vUaTU2P6cOISupzW0P9NBvAMydumNyN1Ohby9MuJPMDSuIuvHE8cn2vC0d0oDc8tzCDc3PPNr8bN/HQ"
    "HDMUMxQzVHuG032lzu5lqmSu5WXqit0ENEGsGJkYmT6sYTnY/HBUDpZ+8KDc8gjnO6GME2UePn02IaNUjgWhg+qhqIaR1Y3q"
    "MoTjmys6rm19vgdVf6yg9FynvP+7MBOcmp/QGXysjUOV48+6UEUHulBFCRmscpopT592sQq3MVYxVjFWMVa1XFMPp3oNX40r"
    "XkNXA92b4CuoFvMV89WHxVew+SFfwdIP5qvVEY7sFTIh751WPdeDaavVKa9EAgNFX+atlER3vuDMNTdOdkJy+F4z6+mqHzSr"
    "cuE6trnkrtOfS+46PI7HsMSwxLBkgaV7SpjVA1XLV8NIfRGbnBruOsxIzEgf8Uyns+OLyqSabyAplbBEHzr3FzPxlzhRMyfR"
    "rYQknTRi3/7U0udvWGCnUqJh/2oaU/sA9GSxtMxuwokuEtGBXnQmJGU8TUjMtroO97GriemJ6YnpqT2Cd1/BG5v71BG7VpCm"
    "oeRNEBROM0AxQL2nkZrUPKauRYdS+LS4HW+gTfugIE3z2RG6jS5K+ngKIrEU1zHRhGot+DDC7fhJHYCaIVmmWW+ZSXWaXNjZ"
    "TiY9YHqhqfMcCT+qXEf6TwiGb0Ekv0dIPk/9Zuhh6GHosbmMHlC+GvzpS1fDPW0BmwAen3mHeed4HUYdJ5GSB+NCOtRhNJ8f"
    "2wSlF/WQPIXqnM1gYZvYhHfb5RSpRFlILoeB3fRx+6wklRwNx3smjHf1EzJidRX1PUUDR9GqmZLk2aYkLdhNxMTExMTEVBPT"
    "/WVudBpSS+Lagby7QjflIGJeYl76wCYhLUciLy0Ojrs0dx5msdyvuA0KipRq1YzeO343sptEHxiqFSOrWedLzFw0nINUf4Wp"
    "tB7pO9UEpOY3rbtduJaZ3XSmF0yAjnRBihKrl8ytZjVKPeqiFN3JMMUwxTDFMNVyP91f80a8Tm29a81YGqreVLABnGemYqZ6"
    "T8fcyJ/UsehQCm/pzpQYGNs+bMztCEN/U8wRrSruQlyV0bVeEqKiK6gVIReogehrqdaELDofYtUZu7cJSeKbS+8PZ/6mAfpx"
    "QupFpFz22agKJrCweZiWzETMRMxEzETteUj3V7dRJ1OlbC0kaunbBAtxnEoGoQ/MubQY+paWB7uWvCOMHpDEF7kyDW/uNIFJ"
    "YHkwOqpaD6L/Rlp2X2rHMaHL7PREqUNS9Ehd9YPWiMxcyzq3WW8vufls1aMoJGMwyjpQh5sYpBikGKQYpFrOpYfVvFGmqvWu"
    "Fdayq3pTC+NmvOkck9WHtmXKzB3ZMWW2Opiu/GPcqfdy//oih00ABLYU/c3zxUWWFJ3YbyIsc9VhQCBM0Lfu15u+QaWwf2sb"
    "zEPKj6rtes3fNJ3SsmmK3/NQ+bxlCjMTMxMzk30R3MMr2cgQXaViLWxqa9nU1HD2RjEzfcxTwxfHxkf/iVdfOq1Wakb31XZM"
    "Zzdvzdv63FqqTytHbxpX/aDZkc6n5uDQeU2nesN06hBzEHMQcxBz0AgH3V6xRp1DtVq11r/1NWtq2I0uYNhh2Dla2Kl2ODnV"
    "b2waxsGkszzGqEmkXqFUAUMi2geSBvPR7YiVX+2hVC2ole3NlFAhsgwHbp9eIDakt/9RJaWXwtU/VRi2mW+L0a3P9eNJqmO9"
    "ZXFnZqiNErVtgKLv5AE3hiaGJoamtvPovuJnjUDZEr7WArkR+ZuMQ0lXMFAxUL2nk7ob35AGq5Z9o83MF3PVJ+Nlals/bIr3"
    "6tj2jaNeQJVAIPO1FM58VkVi01smmUglg3H7OiAb9RDqIrsrSSWKb7Fq+7j6J2mJb+eoAUT1N5Cbz2bDLeQ+HRAU4xPjE+MT"
    "41NrvtLDqd6oO6qleK2d5Pq6Nw1QTE9MTx/YfCV/ZK8TWPqB85UuZv1QTRC6LfXr4XvpoMJVO1jGhtaSzBfVhkqkHGe9vZMS"
    "0yD2r/NI3uCWUttR6gE487dSE8s2vHOvtw0vHej5oaqVc2dWJxTvwssIxQjFCNVBqDuqm93vZJSt5XTq6NsULXm88y7T0nvg"
    "a2rEplIVyt4IKDnIfGPg5HLyaJ9debV/MwAk22a7zhCQnCMCpE58N8+bociCqzq6m5KUYiy2G45OBLCkhOgbS0cRaH5CRFaW"
    "AJZnPUfT2WCkDqlUfiZreIEVB7BkRmJGYkYaD2B5S4EbD13ZiFsLknoSN4FJZ+xTYko6QkpqLBygdDazcNLKPxiT3KPyI5ko"
    "JVrQnGW15APWsYK9xNCd3oLa6uDNm+UiNRr21MNy+m+1D7clBpMzX3RRydFT5NtjcstqVpPNm4TUmZSYlJiUmJQGkZjuqHFW"
    "n1Klb61BuI7KTcASpI1piWnp6GipMvCQhrUXFlbStn8YLHnHNeiGfLyiAiZ7mFPUTuOZ7sR1c3y6btNeS2tc2WrOpDo34WSi"
    "ZGltidlTt/5JwW5tu8X1nEz+wMnUBPd2fJuTyWMnE6MToxOjUyew94Mp3rjXqVG7ltepp3lTUQTY68QcdYRep8bCaYtdm9fJ"
    "O9zr5B8RSNGyXNfzdVy33hYBzmLV2yHg4qZdUVTEEbpPhfquflEwXFuwysXc7W0rp8Ont0N+e34FTIuVbTMUPICRiZGJkYmR"
    "qR1y4A7aNrb7Sa1rrUDfPXWb2kpu7jIaMRodHRo1Jg4yghFb0Ehb/2FstDiMjd6P3eO2uDdU1rDyzLpbkpB1Xm6lqsyMuhL1"
    "NdYVEzpudyshMRrl1PEmqx80qH+nDVFWXjVlybFOWWIyYjJiMmIyas/qvpe2jTqQal1rIKmjbrwdCvPRB8VHtXUDj5a22UoH"
    "olEwWPQGszCIc4jXqDaiAR6ZYIy/CCFFqiCrzSbnHsot0cP5Z+aT6jAPUn+JiNda/ObVi0Mcy3QlnOgtfnMWg6ndNy1+Uzcx"
    "KDEoMSgxKLVA6e4SZ18B5w1WwHVEbmoFnMOzlRiV3jUqKSvDbZSajl+pUzvthbF8ZTq+IpHqW6MTKYAqbEe916DWBsjUwM2A"
    "mpxjo6b2apGlvxBhmQRRtVjEWaLv3P/c0xRUTXwRVWHeJiYsUXo0lKl3Lql/0SRIi3PJ6W0V5/R3ikMi1eDb0uZforuYnJic"
    "mJyYnMYWxd1R5sYDgNcS14oA3hW6qcnevDsc0xPTU01P7hH6nFCiVKnme2y1nOGbazcY4D9zZx1xmZi1pNJ4pO5QDqf6p5IT"
    "W/xvOtNnp2Hsb6RVOZ7cmW3ukr6TCYoJigmKCao94/vOUjcyiaklcy3H00DsJumJY30zPzE/NfzkHcRP47OZ3gk9qbYcqC5j"
    "/4aiUTkzitd2gSI3qrJD95ejnbkr/+Z95J6b3WjOdUr/+uvf1H163K45oPfttqyYozO94Ts60h+/cyqQQno1SH3SH8OjOxmk"
    "GKQYpBikOlvw3k/1GpyyKF57KG+oe1MDejjPTMVMxUxVMZV/hD4p5CpaK9M4W4gAaezfNItzA9UBIMGVUN3V2H4DrSW6aHH2"
    "Eb4z5enWg3nmbzVpwDYxqj8vqodVZ4tBFKdPB9OiGKgYqBioGKg608cfWPBGx/oqsWvgqiN5k9OkmKmYqZipKqZaHOU4X5JQ"
    "54EWJBwXb5ckJnJuiI+4TpA4JLxQR4sbwmC6NM1SzyI3f5OUuJbJUTjR4ye3Pz3KcQ1ALezzyl2eHcUExQTFBNUb27uLvNkj"
    "YBppa0XA7AjcFCy5PCuKaYlpqaal5bHREkU68edOtfHAK7OUN1qjp3JWq46YmGsm5kNRSo/UfXRB/YvW8NomQw3mQg2nQvkq"
    "PoGaSL6yRnE64YlQDEsMSwxL/ThOd1C3kSlQjbI1mNTXtwlQ4ulPDEoMSi1QWh3f9Cf0E1FIhuF4MxRYmOO7S3utk+haptpv"
    "PffFVYyTcnxrcOom9Hn7QJ1j9rLRk57qX/Q1Zoti0A9iMNhypdmdzrcP13EQA+Yn5ifmp+78pweWvdHhukbyWk6orvBNeaE4"
    "sAGzFbOVYatwEA6qqt6DfFDti38VtqrSrVCoTTLmnOIV8zfjCeMJ48lHjCd1z4J3b3dM53T1ZVYbRwjjSi8zmiekLONKvlJl"
    "2bGjquNrxCB//DumCaaJd0wTT40pI1tbsptdpIqracmtrJ6q1la9BRoncmFseZCbAUbU3f2AIhymCKYIpgimiI+MIgwn5NFl"
    "EqEJFNlFHunovdsoYZJgkjgukniW6V3+8AbpulR5oTZ0metIpCZnnxlraWGH4gglcZU4mddKdLXeAiXcQ1BifKyHQYJBgkGC"
    "QeI43RHqREFXoym9wqccHgVzZYpgijg6fwRsYh2ltXNtIwvqMQVBcqFVzXgoWrJFxlPAcPNYN5ZDgcFj3wMjAyMDI8PHhwxV"
    "a9jq4WLqn2Si3hgGVwa7MucBDQaIYwQIq2Wb4u2OY5B1DzuQyqQPRYloMBlC1cbLMHppauMgpOhX4a838fQplSaEIQquUEBb"
    "mZ6KqzK6BmgwcTBxMHEwcRxCHE8zZCzF5erRtAgmCWT6nRTmo63pG86FVXEALkaX0zhdIz1dxrrVd02tabEZwwfDxzuHD50r"
    "Mt0mI/rVkRO6hd6bNhSHyKqmPVUmI5yi6nGKTwYAMeAU57g55QuZa7krMhxLIoh9ILfQZfzJqMKowqjCqPLgqDIlOkwrTCtH"
    "6ypR7wsJhfVvS6V3eNOMeg56XTz7M71jeGrxArYsWr2hysE6vguXuLfhkvE5HO+OSp5HQUQfMw6q6js94eVK5miJjCSMJIwk"
    "jCQPjiRWxWEeYR45Th55ov0kubLsgXfkXFjMOoAYBSXJtV4OG2a3po/LwegNFd/LNHvZFN9BfpFhqR/CIE3E6QBJQcRUzGmU"
    "bRjlZi/tfuTZ6qSGt9VMVVrRiTOkjlS+qEV9xw3xZ5FUHX/W/C1+L7yVJaqaVz26GxikOcyMw4zDjMOMMx5h9j4yZo0zW0lY"
    "E+KjK2QTET5q3ZrEoRv0msGHwed24NOMFSFnpJ1AIJ12pMhH+WCUXNdLY7qPNU1UmW532AjPWGdKqalgBmQ0AiwDNnLeLRs9"
    "MV8Yl1FwpSa5X8XfZVqVqZzjcJyQ6rNVSKG/xMn+bbMDyJWkMk6FK0iras1JlOJKCFaSBTSbuIdLT1un6NJQ1SgejLR2udoI"
    "Fz1LO6Kaq6I22oLSdrKlwhchN/1AteMXMVoxWjFaMVqNo9WD6V/DWQdq3yC0mlHAybC1oxJ3A4Ud0DMwizGL3WlQTOU6iLSB"
    "0cYWo/b1WU1Wg/e4E2q5t0Ot8WGwu4LW/9CrptmOvM0wlKIq0kuIZT4eqjaQ1fih5wvUS6piNnZCMpovt5W69gbnExLBd5oO"
    "6m/+hnS4Cws94UTz0C4zdU8xKTEpMSkxKVkiz95exay+p0rBGgbq6tgEA3Uk6wbymZZqhh6GntvGbqlG1krVHpL4Em+n109J"
    "srNQtrI4FghOB3D5pmeXMao9PZSE/h/FKgJO0nwBAA=="
)
if not _RUTA_DIAGNOSTICO.exists():
    _RUTA_DIAGNOSTICO.parent.mkdir(parents=True, exist_ok=True)
    _RUTA_DIAGNOSTICO.write_bytes(gzip.decompress(base64.b64decode(_BLOB_DIAGNOSTICO)))
    print(f"Escrito {_RUTA_DIAGNOSTICO} (copia embebida).")
diagnostico = [json.loads(l) for l in _RUTA_DIAGNOSTICO.read_text(encoding="utf-8").splitlines()]
_sha = hashlib.sha256(_RUTA_DIAGNOSTICO.read_bytes()).hexdigest()
print(f"data/trayectorias_diagnostico.jsonl: {len(diagnostico)} registros | SHA coincide: {_sha == _SHA_DIAGNOSTICO}")

In [ ]:
from collections import Counter

train = [t for t in diagnostico if t["split"] == "train"]
dev = [t for t in diagnostico if t["split"] == "dev"]
print(f"train: {len(train)}  |  dev: {len(dev)}")
print("por familia:", dict(Counter(t["familia"] for t in diagnostico)))
print("paso señalado:", dict(sorted(Counter(
    t["paso_error"] for t in diagnostico if t["paso_error"]).items())))
print("\nUn ejemplo completo, lo que el modelo tiene que aprender a escribir:\n")
print(train[0]["mensajes"][-1]["content"])
print("\n" + "-" * 70)
print("Y el prompt que lo precede (el MISMO de inferencia, `prompt_revision`):\n")
print(train[0]["mensajes"][1]["content"][:700], "…")

---
## 5 · El modelo y los adaptadores que ya existen

In [ ]:
import json, time
from contextlib import nullcontext
from pathlib import Path

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

MODELO_BASE    = "Qwen/Qwen2.5-1.5B-Instruct"
DIR_LORA_M1    = "adaptadores/qwen-lora"          # M1: formato "Paso N / Respuesta final"
DIR_LORA_AGENT = "adaptadores/qwen-lora-agente"   # S10: entrenado con trayectorias (opcional)

if not Path(DIR_LORA_M1).exists():
    raise FileNotFoundError(f"No se encuentra {DIR_LORA_M1}. Ejecuten S04_Lab_Fine_tuning_Qwen.ipynb "
                            "o monten el Drive donde quedó guardado.")

tok = AutoTokenizer.from_pretrained(MODELO_BASE)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

_base = AutoModelForCausalLM.from_pretrained(MODELO_BASE, torch_dtype=torch.float16).to(DEVICE)
modelo = PeftModel.from_pretrained(_base, DIR_LORA_M1, adapter_name="m1").eval()
ADAPTADORES = ["m1"]
if Path(DIR_LORA_AGENT).exists():
    modelo.load_adapter(DIR_LORA_AGENT, adapter_name="agente")
    ADAPTADORES.append("agente")
print("Adaptadores cargados:", ADAPTADORES, "(+ None = modelo base)")


class ContadorTokens:
    """Acumula tokens y llamadas al LLM de la pregunta en curso."""
    def __init__(self):
        self.reset()

    def reset(self):
        self.llamadas, self.entrada, self.salida = 0, 0, 0


CONTADOR = ContadorTokens()
_CACHE_LLM = {}


@torch.no_grad()
def generar_chat(mensajes, esquemas=None, adaptador="m1", max_new_tokens=220, muestreo=None):
    """Una generación. `adaptador`: "m1", "agente" o None (modelo base).
    `muestreo`: None (greedy) o (temperatura, semilla) para autoconsistencia."""
    clave = (json.dumps(mensajes, ensure_ascii=False, sort_keys=True, default=str),
             json.dumps(esquemas, sort_keys=True) if esquemas else None, adaptador, max_new_tokens, muestreo)
    prompt = tok.apply_chat_template(mensajes, tools=esquemas, tokenize=False, add_generation_prompt=True)
    ids = tok(prompt, return_tensors="pt", add_special_tokens=False).to(modelo.device)
    CONTADOR.llamadas += 1
    CONTADOR.entrada += ids["input_ids"].shape[1]
    if clave in _CACHE_LLM:
        texto, n_out = _CACHE_LLM[clave]
        CONTADOR.salida += n_out
        return texto
    if adaptador is None:
        contexto = modelo.disable_adapter()
    else:
        modelo.set_adapter(adaptador)
        contexto = nullcontext()
    kwargs = dict(max_new_tokens=max_new_tokens, pad_token_id=tok.pad_token_id, eos_token_id=tok.eos_token_id)
    if muestreo is None:
        kwargs["do_sample"] = False
    else:
        temperatura, semilla = muestreo
        torch.manual_seed(semilla)
        kwargs.update(do_sample=True, temperature=temperatura, top_p=0.95)
    with contexto:
        out = modelo.generate(**ids, **kwargs)
    nuevos = out[0][ids["input_ids"].shape[1]:]
    texto = tok.decode(nuevos, skip_special_tokens=True).strip()
    _CACHE_LLM[clave] = (texto, len(nuevos))
    CONTADOR.salida += len(nuevos)
    return texto


def crear_llm(adaptador="m1", max_new_tokens=220, muestreo=None):
    """La firma que espera el núcleo del agente: llm(mensajes, esquemas) -> texto."""
    def llm(mensajes, esquemas):
        return generar_chat(mensajes, esquemas, adaptador=adaptador,
                            max_new_tokens=max_new_tokens, muestreo=muestreo)
    return llm

---
## 6 · Tokenización

Gradiente **solo** sobre el turno del asistente: lo que se enseña es a
dictaminar, no a reproducir el prompt.

In [ ]:
MAX_LEN = 1536      # el prompt de revisión es corto: ~400-600 tokens


def render(mensajes, generacion=False):
    return tok.apply_chat_template(mensajes, tokenize=False,
                                   add_generation_prompt=generacion)


def tokenizar(t):
    """Gradiente SOLO sobre el turno del asistente: lo que se enseña es a
    dictaminar, no a reproducir el prompt."""
    mensajes = t["mensajes"]
    completo = render(mensajes)
    antes = render(mensajes[:-1], generacion=True)
    assert completo.startswith(antes), "la plantilla no es prefijo-estable"
    enc = tok(completo, return_offsets_mapping=True, add_special_tokens=False,
              truncation=True, max_length=MAX_LEN)
    etiquetas = [tid if a >= len(antes) else -100
                 for tid, (a, _) in zip(enc["input_ids"], enc["offset_mapping"])]
    return {"input_ids": enc["input_ids"], "attention_mask": enc["attention_mask"],
            "labels": etiquetas, "truncado": len(enc["input_ids"]) >= MAX_LEN}


tk = tokenizar(train[0])
print(f"Tokens: {len(tk['input_ids'])} | con gradiente: {sum(e != -100 for e in tk['labels'])}")
print("\nLo que el modelo aprende a escribir (solo los tokens con etiqueta):\n")
print(tok.decode([t for t, e in zip(tk["input_ids"], tk["labels"]) if e != -100]))

In [ ]:
from datasets import Dataset

tok_train = [tokenizar(t) for t in train]
tok_dev = [tokenizar(t) for t in dev]
longitudes = [len(t["input_ids"]) for t in tok_train]
print(f"tokens por ejemplo: min {min(longitudes)}  mediana {int(np.median(longitudes))}  "
      f"max {max(longitudes)}")
assert not any(t["truncado"] for t in tok_train + tok_dev), "hay ejemplos truncados: suban MAX_LEN"

columnas = ["input_ids", "attention_mask", "labels"]
ds_train = Dataset.from_list([{k: t[k] for k in columnas} for t in tok_train])
ds_dev = Dataset.from_list([{k: t[k] for k in columnas} for t in tok_dev])
print(f"train {len(ds_train)} | dev {len(ds_dev)}")

---
## 7 · Línea base, antes de entrenar

Sin esto, "mejoró" no significaría nada. Se mide el modelo base y los dos
adaptadores que ya existen sobre los **mismos** casos de desarrollo.

La columna `reescribe` cuenta las salidas que contienen `Respuesta final`: es
exactamente la conducta que se quiere quitar.

In [ ]:
# La medición, separando lo que no se debe mezclar.
#
#   formato   -> ¿emite una línea `Diagnóstico:` con una de las cinco categorías?
#   acierto   -> ¿esa categoría es la que el caso espera?
#
# Con 33 ejemplos de entrenamiento lo esperable es que suba el formato. Que suba
# el acierto es otra cosa, y presentarlas juntas haría pasar por razonamiento lo
# que puede ser obediencia al formato.
@torch.no_grad()
def medir(casos, adaptador, nombre):
    filas = []
    for t in casos:
        prompt = tok.apply_chat_template(t["mensajes"][:-1], tokenize=False,
                                         add_generation_prompt=True)
        ids = tok(prompt, return_tensors="pt", add_special_tokens=False).to(modelo.device)
        if adaptador is None:
            contexto = modelo.disable_adapter()
        else:
            modelo.set_adapter(adaptador)
            contexto = nullcontext()
        with contexto:
            out = modelo.generate(**ids, max_new_tokens=120, do_sample=False,
                                  pad_token_id=tok.pad_token_id)
        salida = tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        conclusion = extraer_diagnostico(salida)
        filas.append({
            "id": t["id"], "familia": t["familia"], "esperado": t["diagnostico"].lower(),
            "conclusion": conclusion,
            "formato_ok": diagnostico_valido(conclusion),
            "acierto": diagnostico_valido(conclusion) and conclusion == t["diagnostico"].lower(),
            # La conducta que se quiere quitar: reescribir el procedimiento.
            "reescribe": "Respuesta final" in salida,
            "salida": salida[:300],
        })
    df = pd.DataFrame(filas)
    print(f"{nombre:28s} formato {int(df.formato_ok.sum())}/{len(df)}   "
          f"acierto {int(df.acierto.sum())}/{len(df)}   "
          f"reescribe {int(df.reescribe.sum())}/{len(df)}")
    return df

In [ ]:
# LÍNEA BASE, antes de entrenar. Sin esto no se sabría si el entrenamiento hizo
# algo, y "mejoró" no significaría nada.
BASES = {}
for adaptador, nombre in [(None, "base (sin adaptador)"), ("m1", "qwen-lora (M1)"),
                          ("agente", "qwen-lora-agente (S10)")]:
    if adaptador is not None and adaptador not in ADAPTADORES:
        print(f"{nombre:28s} no está en Drive, se omite")
        continue
    BASES[nombre] = medir(dev, adaptador, nombre)

print("\n`agente` es la referencia que importa: es el adaptador con el que se")
print("ejecutaron las tres corridas de M4.")

---
## 8 · Entrenamiento

### Weights & Biases

W&B registra automáticamente la curva de pérdida, los hiperparámetros y el
consumo de GPU. Es lo que después nos permitirá comparar las cuatro
arquitecturas sobre los mismos ejes en lugar de sobre capturas de pantalla.

Convención de nombres del proyecto (idéntica en los cinco notebooks):

- **Proyecto:** `tutor-matematicas-arquitecturas`
- **Run:** `qwen-lora-diagnostico`
- **Tags:** identifican arquitectura y fase, para poder filtrar en el panel

Si no quieren usar W&B, pongan `USAR_WANDB = False`: el notebook seguirá
funcionando y las métricas se guardarán igual en `resultados/`.

In [ ]:
import os

USAR_WANDB   = True          # ponlo en False para trabajar sin conexión a W&B
PROYECTO     = "tutor-matematicas-arquitecturas"
NOMBRE_RUN   = "qwen-lora-diagnostico"
TAGS         = ["m4", "diagnostico", "vision", "finetuning"]

if USAR_WANDB:
    import wandb
    wandb.login()            # pedirá la API key la primera vez
    os.environ["WANDB_PROJECT"] = PROYECTO
    os.environ["WANDB_LOG_MODEL"] = "false"
    REPORTAR_A = "wandb"
else:
    os.environ["WANDB_MODE"] = "disabled"
    REPORTAR_A = "none"

print(f"Registro de experimentos: {REPORTAR_A}  |  run: {NOMBRE_RUN}")

In [ ]:
from peft import LoraConfig, get_peft_model
from transformers import DataCollatorForSeq2Seq, Trainer

DIR_ADAPTADOR = "adaptadores/qwen-lora-diagnostico"     # NUEVO, no toca los otros dos
assert DIR_ADAPTADOR not in ("adaptadores/qwen-lora", "adaptadores/qwen-lora-agente")

# Se entrena sobre el modelo BASE, no encima de otro adaptador: así el experimento
# aísla el efecto de estos 33 ejemplos y los otros dos adaptadores quedan intactos.
base_limpio = AutoModelForCausalLM.from_pretrained(
    MODELO_BASE, torch_dtype=torch.float16).to(DEVICE)
base_limpio.gradient_checkpointing_enable()
base_limpio.enable_input_require_grads()
base_limpio.config.use_cache = False

config_lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
entrenable = get_peft_model(base_limpio, config_lora)
# LoRA en fp32 sobre pesos base en fp16: en fp16 el gradiente se desborda (lección de M1).
for n, p in entrenable.named_parameters():
    if p.requires_grad:
        p.data = p.data.float()
entrenable.print_trainable_parameters()

args = construir_args(
    output_dir=f"{DIR_CHECKPOINTS}/qwen-lora-diagnostico",
    num_train_epochs=6,              # 33 ejemplos: hacen falta más pasadas que en S10
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=2,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    fp16=True,
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to=REPORTAR_A,
    run_name=NOMBRE_RUN,
    seed=SEMILLA,
)
trainer = Trainer(model=entrenable, args=args, train_dataset=ds_train, eval_dataset=ds_dev,
                  data_collator=DataCollatorForSeq2Seq(tokenizer=tok, padding=True,
                                                       label_pad_token_id=-100))
resultado = trainer.train()
print(resultado)

entrenable.save_pretrained(DIR_ADAPTADOR)
print(f"\nAdaptador guardado en {DIR_ADAPTADOR}")
print("Los otros dos siguen donde estaban:", [d for d in ADAPTADORES])

---
## 9 · Después de entrenar · formato y acierto, separados

In [ ]:
# El adaptador nuevo, cargado junto a los otros para medir en las mismas condiciones.
del entrenable, base_limpio
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

modelo.load_adapter(DIR_ADAPTADOR, adapter_name="diagnostico")
ADAPTADORES.append("diagnostico")
DESPUES = medir(dev, "diagnostico", "qwen-lora-diagnostico")

print("\n" + "=" * 78)
print("ANTES Y DESPUÉS · sobre los casos de DESARROLLO, no sobre los 12 de evaluación")
print("=" * 78)
print(f"{'adaptador':28s}{'formato':>10s}{'acierto':>10s}{'reescribe':>11s}")
for nombre, df in list(BASES.items()) + [("qwen-lora-diagnostico", DESPUES)]:
    print(f"{nombre:28s}{int(df.formato_ok.sum()):>7d}/{len(df):<3d}"
          f"{int(df.acierto.sum()):>7d}/{len(df):<3d}{int(df.reescribe.sum()):>8d}/{len(df):<3d}")
print("=" * 78)

n = len(DESPUES)
print(f"\nSon {n} casos de desarrollo. Con ese tamaño:")
print("  · una diferencia de 1 o 2 casos no distingue nada;")
print("  · el formato y el acierto NO se promedian ni se presentan como una cifra;")
print("  · y esto no dice nada todavía sobre los 12 casos de evaluación, que no se")
print("    han tocado aquí. Eso lo mide S13.")

if int(DESPUES.formato_ok.sum()) < n:
    print("\nEl formato no se sostiene ni después de entrenar. Antes de volver a S13,")
    print("mirar una salida completa: puede que 33 ejemplos no basten, o que el")
    print("prompt y las etiquetas no coincidan tanto como parecía.")

---
## 10 · Artefactos

In [ ]:
import json
from pathlib import Path

Path("m4").mkdir(exist_ok=True)
resumen = {
    "adaptador": DIR_ADAPTADOR,
    "no_sobrescribe": ["adaptadores/qwen-lora", "adaptadores/qwen-lora-agente"],
    "datos": {"train": len(train), "dev": len(dev),
              "familias": dict(Counter(t["familia"] for t in diagnostico))},
    "lora": {"r": config_lora.r, "alpha": config_lora.lora_alpha,
             "target_modules": sorted(config_lora.target_modules)},
    "entrenamiento": {"epocas": float(args.num_train_epochs), "lr": args.learning_rate,
                      "loss_final": float(resultado.training_loss)},
    "dev_antes": {n: {"formato": int(d.formato_ok.sum()), "acierto": int(d.acierto.sum()),
                      "reescribe": int(d.reescribe.sum()), "n": len(d)}
                  for n, d in BASES.items()},
    "dev_despues": {"formato": int(DESPUES.formato_ok.sum()),
                    "acierto": int(DESPUES.acierto.sum()),
                    "reescribe": int(DESPUES.reescribe.sum()), "n": len(DESPUES)},
    "aviso": ("Medido SOLO sobre los casos de desarrollo. Los 12 casos de "
              "evaluación de M4 no se tocaron en este notebook."),
}
json.dump(resumen, open("m4/finetuning_diagnostico.json", "w", encoding="utf-8"),
          ensure_ascii=False, indent=2)
DESPUES.to_csv("m4/dev_diagnostico_despues.csv", index=False)
for nombre, df in BASES.items():
    df.to_csv(f"m4/dev_diagnostico_{nombre.split()[0].replace('(', '')}.csv", index=False)

print("Guardado:")
print("  m4/finetuning_diagnostico.json")
print("  m4/dev_diagnostico_despues.csv  (y uno por cada línea base)")
print(f"  {DIR_ADAPTADOR}/   <- súbanlo igual que los otros adaptadores")

---
## 11 · Cómo leer esto, y qué hacer después

**1 · Mirar primero `reescribe`.** Si baja de golpe, el entrenamiento consiguió lo
que buscaba: que el modelo deje de rehacer el ejercicio. Es el cambio de conducta
que bloqueaba M4.

**2 · Formato y acierto, separados y en ese orden.** Con 33 ejemplos, el formato
es lo que puede moverse. Si sube el formato y el acierto se queda, el resultado
honesto es "ahora dictamina, pero todavía no localiza bien", que es información
útil y no una mejora que presentar como tal.

**3 · Son 9 casos de desarrollo.** Una diferencia de uno o dos no distingue nada.

**4 · Esto no dice nada sobre los 12 casos de evaluación.** No se han tocado. El
siguiente paso es S13 con `ADAPTADOR_DIAGNOSTICO = "diagnostico"`, y ahí la
puerta de desarrollo decide si la evaluación llega a ejecutarse.

**5 · Si el formato no se sostiene**, las salidas a mirar son las de
`m4/dev_diagnostico_despues.csv`. Las dos causas probables son que 33 ejemplos no
basten —hay que escribir más, no reciclar los 12 casos— o que el prompt de
entrenamiento se haya separado del de inferencia, que es justo lo que
`prompt_revision` existe para evitar.